# RSNA Knee MRI — frozen DINOv2 + finding-specific attention training

A self-contained **training** notebook for Kaggle. It reads our normalized MRI dataset directly, caches frozen DINOv2 Small features, trains fold-specific attention heads and exports a model package. It does not submit predictions, publish datasets, generate new report labels, or fine-tune the image encoder.

## Attach these inputs

1. **`gany24558/rsna-knee-normalized-training-data`**, latest complete version. All active shards must be processed. Existing labels/masks/weights/folds are inside these archives; no separate Qwen dataset is needed.
2. The competition dataset, for `train.csv` verification. Training images come from the normalized dataset, not the original DICOMs.
3. A **generic public DINOv2 Small, without registers** checkpoint. Supported: a Hugging Face `facebook/dinov2-small` directory with `config.json` and local weights, or Meta's official `dinov2_vits14_pretrain.pth` from an attached Kaggle model/dataset. No task-fine-tuned fold encoder. An arbitrary `.pth` architecture is not interchangeable.
4. Optional: previous **private** notebook outputs for feature/checkpoint resumption.

Select a GPU (one T4/P100 is sufficient for the design; runtime must be measured), leave **Internet off**, and use **Save Version → Run All**. Kaggle usually provides the required packages. There are no online `pip install`, model download, API credential or upload calls. If a package is missing, attach an offline wheel dataset or select a compatible Kaggle image.

## Rules translated into implementation

Reviewed project snapshots: `docs/rules.md`, `docs/data.md`, `docs/overview.md` (2026-09-24). External publicly accessible pretrained models are allowed; training does not read test labels or manually label test/validation records. The overview's **9-hour / Internet-off / submission.csv** requirements apply to the later submission notebook. This training run uses a conservative 7.5-hour budget and resumes across sessions.

Competition data redistribution is restricted. Keep this notebook and its complete output **private**: cached features, study IDs, training labels and OOF predictions are derived competition data. Only the separate `model_package` is prepared without those records; review the source rules and model license before distributing it. No automatic publishing occurs.

This is an implementable baseline, not a reproduced leaderboard score. The small verified set has already informed label development, so cross-validation is development evidence rather than an untouched final evaluation.

## 1. Configuration
The default runs feature extraction followed by all five folds. Use `features` or `train` to separate sessions. Hyperparameter changes create a separate training run; compatible frozen feature caches can still be reused.

In [ ]:
from pathlib import Path
DATASET_ROOT = None      # Example: Path('/kaggle/input/datasets/gany24558/rsna-knee-normalized-training-data')
COMPETITION_ROOT = None  # Directory containing the original train.csv and train_series.csv
ENCODER_PATH = None      # HF folder OR path to dinov2_vits14_pretrain.pth; auto-discovery fails on ambiguity
ENCODER_BACKEND = 'auto' # 'auto', 'hf', or 'timm'
RESUME_ROOTS = []        # Previous run folders containing run_identity.json; otherwise discovered in /kaggle/input
OUTPUT_ROOT = Path('/kaggle/working/rsna-knee-training')
MODE = 'all'            # 'all', 'features', 'train'
FOLDS = [0, 1, 2, 3, 4] # For a pilot, use [0]; export is then clearly marked partial OOF
MAX_HOURS = 7.5
REQUIRE_GPU = True
CFG = dict(seed=42, hidden=256, lr=7e-4, weight_decay=2e-3,
           batch_size=64, epochs=32, patience=5, centers=16,
           verified_fraction=0.25, series_dropout=0.10, ema_decay=0.995,
           rank_lambda=0.0,     # Controlled experiment: 0.05 adds verified-only ranking loss
           refine_epochs=0,    # Controlled experiment: up to 5 verified-only epochs
           refine_lr=1e-4, image_batch=8, representation='single', # 'single' or 'triplet'
           min_free_gb=2.0, max_feature_ram_gb=5.0)


## 2. Runtime imports and reproducibility
Definitions below are embedded in this notebook; no repository checkout is required. The exported runtime is assembled from these exact executed definitions.

In [ ]:
"""Offline RSNA frozen-DINO training runtime. Embedded verbatim in the notebook."""
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import copy, gc, hashlib, io, json, math, random, shutil, tarfile, time
from contextlib import nullcontext
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

ID, SID = 'StudyInstanceUID', 'SeriesInstanceUID'
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
RUNTIME_VERSION = 'rsna-frozen-dino-v1'




### Utilities and session budget

In [ ]:
def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(data, indent=2, sort_keys=True, allow_nan=False, default=str))
    tmp.replace(path)


def atomic_torch(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp'); torch.save(data, tmp); tmp.replace(path)


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


class SessionBudget:
    def __init__(self, hours):
        self.start = time.monotonic(); self.seconds = hours * 3600
    def expired(self, reserve=90):
        return time.monotonic() - self.start > self.seconds - reserve


def select_one(candidates, description):
    candidates = sorted(set(Path(p).resolve() for p in candidates))
    if len(candidates) != 1:
        raise ValueError(f'Set {description} explicitly: found {len(candidates)} candidates: {candidates}')
    return candidates[0]


def find_input_files(root, filename):
    """Discover lightweight manifests without traversing hundreds of thousands of DICOMs."""
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in {'train_series','test_series','cache','features','.git','__pycache__'}]
        if filename in files:
            yield Path(current)/filename


### Published-dataset loader and label audit

In [ ]:
def discover_bundle(explicit=None, input_root='/kaggle/input'):
    if explicit:
        root = Path(explicit)
    else:
        candidates = []
        for p in find_input_files(input_root,'dataset_index.json'):
            try:
                index = json.loads(p.read_text())
                if index.get('dataset_handle') == 'gany24558/rsna-knee-normalized-training-data':
                    candidates.append(p.parent)
            except (ValueError, OSError): pass
        root = select_one(candidates, 'DATASET_ROOT')
    if not (root / 'dataset_index.json').is_file():
        raise FileNotFoundError('Attach gany24558/rsna-knee-normalized-training-data, with dataset_index.json')
    return root


def safe_member(archive, name):
    part = PurePosixPath(name)
    if part.is_absolute() or '..' in part.parts:
        raise ValueError(f'Unsafe archive member: {name}')
    member = archive.getmember(name)
    if not member.isfile(): raise ValueError(f'Not a regular member: {name}')
    return member


def load_bundle(root):
    """Use only active, processed shards; never legacy archives or partial snapshots."""
    root = Path(root); index = json.loads((root / 'dataset_index.json').read_text())
    identity = index['identity']; expected = {f'{i:03d}' for i in range(int(identity['num_shards']))}
    if set(index['shards']) != expected:
        raise ValueError(f'Preprocessing is incomplete. Missing shards: {sorted(expected-set(index["shards"]))}')
    frames, raw_reference, labels, prep_reference = [], None, None, None
    for slot, entry in sorted(index['shards'].items()):
        if entry['status'] != 'processed': raise ValueError(f'Shard {slot} is partial; resume preprocessing first')
        if Path(entry['file']).name != entry['file']: raise ValueError('Invalid package filename')
        package = root / entry['file']
        with tarfile.open(package, 'r:') as arc:
            info = json.load(arc.extractfile(safe_member(arc, 'publication_manifest.json')))
            if info['status'] != 'processed' or {k: info[k] for k in identity} != identity:
                raise ValueError('Publication identity/status mismatch')
            prep = json.load(arc.extractfile(safe_member(arc, 'preprocessing_config.json')))
            if prep.get('run_id') != identity['run_id']: raise ValueError('Preprocessing config run mismatch')
            if prep_reference is not None and prep != prep_reference: raise ValueError('Preprocessing configs differ')
            prep_reference = prep
            raw = arc.extractfile(safe_member(arc, 'study_labels_and_folds.csv')).read()
            if hashlib.sha256(raw).hexdigest() != identity['label_sha256']:
                raise ValueError('Label checksum mismatch')
            if raw_reference is not None and raw != raw_reference: raise ValueError('Labels differ across shards')
            raw_reference = raw
            if labels is None:
                labels = pd.read_csv(io.BytesIO(raw), dtype={ID: str, 'patient_group': str})
            frame = pd.read_csv(arc.extractfile(safe_member(arc, f'training_series_shard_{int(slot):03d}.csv')),
                                dtype={ID: str, SID: str})
            if len(frame) and not frame.status.eq('ok').all(): raise ValueError('Training series contain failures')
            # Direct seek offsets avoid repeatedly scanning tar archives during extraction.
            offsets, sizes = [], []
            for member in frame.cache_path:
                item = safe_member(arc, member); offsets.append(item.offset_data); sizes.append(item.size)
            frame['archive_path'] = str(package); frame['offset'] = offsets; frame['nbytes'] = sizes
            frames.append(frame)
    series = pd.concat(frames, ignore_index=True).sort_values([ID, SID]).reset_index(drop=True)
    if series.empty or series.duplicated([ID, SID]).any(): raise ValueError('Empty or duplicated training series')
    if not labels[ID].is_unique: raise ValueError('Duplicate study labels')
    if not set(series[ID]).issubset(set(labels[ID])): raise ValueError('Series with no label record')
    validate_labels(labels)
    eligible = labels.loc[labels[ID].isin(series[ID])].copy().sort_values(ID).reset_index(drop=True)
    index = dict(index, preprocessing=prep_reference)
    return series, eligible, labels, index


def validate_labels(labels):
    required = {ID, 'fold', 'patient_group', 'has_gold'}
    required |= {t+s for t in TARGETS for s in ['', '__mask', '__weight', '__gold']}
    if required - set(labels): raise ValueError(f'Missing label columns: {sorted(required-set(labels))}')
    if labels.patient_group.isna().any() or labels.patient_group.str.strip().eq('').any():
        raise ValueError('Missing grouping identity')
    folds = pd.to_numeric(labels.fold, errors='raise').to_numpy(float)
    if not np.isfinite(folds).all() or not np.equal(folds, folds.astype(int)).all() or (folds < -1).any():
        raise ValueError('Invalid fold values')
    if labels.groupby('patient_group').fold.nunique().gt(1).any(): raise ValueError('A group crosses folds')
    for t in TARGETS:
        for s in ['__mask', '__gold']:
            if not labels[t+s].isin([0, 1]).all(): raise ValueError('Invalid label mask')
        m = labels[t+'__mask'].eq(1); g = labels[t+'__gold'].eq(1)
        w = labels[t+'__weight'].to_numpy(float)
        if not np.isfinite(w).all() or (w < 0).any(): raise ValueError('Invalid reliability weights')
        if not labels.loc[m, t].isin([0, 1]).all(): raise ValueError('Known target is not binary')
        if (g & ~m).any() or (g & labels[t+'__weight'].ne(1)).any(): raise ValueError('Invalid verified override')
    any_gold = labels[[t+'__gold' for t in TARGETS]].any(axis=1)
    if not labels.has_gold.eq(any_gold.astype(int)).all(): raise ValueError('has_gold does not match per-target masks')
    if labels.loc[any_gold, 'fold'].lt(0).any(): raise ValueError('Verified studies need validation folds')


def audit_labels(labels):
    rows = []
    for fold, block in labels.groupby('fold'):
        for t in TARGETS:
            m = block[t+'__mask'].eq(1) & block[t+'__weight'].gt(0)
            g = block[t+'__gold'].eq(1)
            rows.append(dict(fold=int(fold), target=t, studies=len(block), positive=int((m & block[t].eq(1)).sum()),
                             negative=int((m & block[t].eq(0)).sum()), unknown=int((~m).sum()),
                             verified_positive=int((g & block[t].eq(1)).sum()),
                             verified_negative=int((g & block[t].eq(0)).sum())))
    return pd.DataFrame(rows)


### Normalized-series validation and offline encoder

In [ ]:
def read_native(record, expected_run):
    with open(record['archive_path'], 'rb') as f:
        f.seek(int(record['offset'])); payload = f.read(int(record['nbytes']))
    if len(payload) != int(record['nbytes']): raise ValueError('Truncated archive')
    with np.load(io.BytesIO(payload), allow_pickle=False) as z:
        image = z['image']; meta = json.loads(str(z['meta'].item()))
        support = np.unpackbits(z['valid_bits'], count=image.size).reshape(image.shape).astype(bool)
    if image.dtype != np.float16 or image.shape != (64, 320, 320) or list(image.shape) != meta['shape']:
        raise ValueError('Expected preprocessing contract float16 [64,320,320]')
    if meta.get('representation') != 'native-plane-full-fov-v3' or meta.get('run_id') != expected_run:
        raise ValueError('Wrong normalization identity/representation')
    if meta.get(ID) != record[ID] or meta.get(SID) != record[SID]: raise ValueError('Cache ID mismatch')
    if not np.isfinite(image).all() or image.min() < 0 or image.max() > 1: raise ValueError('Invalid pixel range')
    n = int(meta['selected_count'])
    if n <= 0 or n > len(image) or support[n:].any(): raise ValueError('Invalid acquired-slice mask')
    valid = np.flatnonzero(support[:n].reshape(n, -1).any(axis=1))
    positions = np.asarray(meta['slice_positions_mm'], dtype=np.float32)
    spacing = np.asarray(meta['spacing_summary_drc_mm'], dtype=np.float32)
    if len(valid) != n or len(positions) != n or not np.isfinite(positions).all(): raise ValueError('Invalid slice positions')
    if n > 1 and not (np.diff(positions) > 0).all(): raise ValueError('Unordered slice positions')
    if spacing.shape != (3,) or not np.isfinite(spacing).all() or (spacing <= 0).any(): raise ValueError('Invalid spacing')
    return image, support, meta


def discover_encoder(explicit=None, backend='auto', input_root='/kaggle/input'):
    if explicit:
        p = Path(explicit)
        chosen = ('hf' if p.is_dir() else 'timm') if backend == 'auto' else backend
        return p, chosen
    candidates = []
    for p in find_input_files(input_root,'config.json'):
        try:
            c = json.loads(p.read_text())
            if c.get('model_type') == 'dinov2' and c.get('hidden_size') == 384:
                if any(p.parent.glob('*.safetensors')) or (p.parent/'pytorch_model.bin').exists():
                    candidates.append((p.parent, 'hf'))
        except (ValueError, OSError): pass
    for p in find_input_files(input_root,'dinov2_vits14_pretrain.pth'):
        candidates.append((p, 'timm'))
    if backend != 'auto': candidates = [(p,b) for p,b in candidates if b == backend]
    if len(candidates) != 1: raise ValueError(f'Set ENCODER_PATH/BACKEND explicitly. Found: {candidates}')
    return candidates[0]


def encoder_source_hash(path):
    path = Path(path)
    if path.is_file(): return file_hash(path)
    selected = sorted(p for p in path.iterdir() if p.is_file() and
                      (p.suffix in {'.json', '.safetensors'} or p.name == 'pytorch_model.bin'))
    if not selected: raise ValueError('Empty encoder folder')
    return fingerprint({p.name: file_hash(p) for p in selected})


class FrozenEncoder(nn.Module):
    def __init__(self, path, backend, size=322):
        super().__init__(); self.backend = backend; self.size = size
        if backend == 'hf':
            from transformers import AutoModel
            self.encoder = AutoModel.from_pretrained(str(path), local_files_only=True, trust_remote_code=False)
            c = self.encoder.config
            if c.model_type != 'dinov2' or c.hidden_size != 384 or c.patch_size != 14 or c.num_hidden_layers != 12:
                raise ValueError('Attach the generic DINOv2 Small without registers or a classifier')
        elif backend == 'timm':
            import timm
            from timm.models.vision_transformer import checkpoint_filter_fn
            self.encoder = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=False,
                                              num_classes=0, img_size=size)
            state = torch.load(path, map_location='cpu', weights_only=True)
            state = checkpoint_filter_fn(state, self.encoder)
            self.encoder.load_state_dict(state, strict=True)
        else: raise ValueError('Unknown encoder backend')
        self.requires_grad_(False); self.eval()
        self.register_buffer('mean', torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std', torch.tensor([.229,.224,.225]).view(1,3,1,1))

    def forward(self, images, support):
        # Deliberately no crop/rescale: preserve 320px cache, add a one-pixel border.
        if images.shape[1:] != (3,320,320): raise ValueError('Unexpected encoder input')
        images = F.pad(images, (1,1,1,1)); support = F.pad(support, (1,1,1,1))
        x = (images-self.mean)/self.std
        tokens = self.encoder(pixel_values=x).last_hidden_state if self.backend=='hf' else self.encoder.forward_features(x)
        if tokens.shape[1:] != (530,384): raise ValueError('Unexpected DINO token layout')
        weights = F.avg_pool2d(support.float(),14,14).flatten(1)
        pooled = (tokens[:,1:].float()*weights.unsqueeze(-1)).sum(1)/weights.sum(1,keepdim=True).clamp_min(1e-6)
        return torch.cat([tokens[:,0].float(),pooled],1)

    @classmethod
    def from_export(cls, folder):
        folder = Path(folder); adapter = json.loads((folder/'adapter.json').read_text())
        if adapter['backend'] == 'hf': return cls(folder,'hf',adapter['size'])
        # Export is already converted/resized: use the exact architecture, no checkpoint filtering.
        import timm
        model = cls.__new__(cls); nn.Module.__init__(model)
        model.backend = 'timm'; model.size = adapter['size']
        model.encoder = timm.create_model(adapter['architecture'],pretrained=False,num_classes=0,img_size=model.size)
        model.encoder.load_state_dict(torch.load(folder/'encoder.pt',map_location='cpu',weights_only=True),strict=True)
        model.register_buffer('mean',torch.tensor(adapter['mean']).view(1,3,1,1))
        model.register_buffer('std',torch.tensor(adapter['std']).view(1,3,1,1))
        model.requires_grad_(False);model.eval()
        return model

    def export(self, folder):
        folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
        if self.backend == 'hf': self.encoder.save_pretrained(folder, safe_serialization=True)
        else: atomic_torch(folder/'encoder.pt', self.encoder.state_dict())
        atomic_json(folder/'adapter.json',dict(backend=self.backend,size=self.size,
                    architecture='vit_small_patch14_dinov2.lvd142m',converted_state=True,
                    mean=[.485,.456,.406],std=[.229,.224,.225],patch_size=14))


### Resumable feature extraction

In [ ]:
def make_slice_inputs(image, support, centers, representation='single'):
    n = int(support.reshape(len(support),-1).any(1).sum())
    if representation == 'single': indices = np.repeat(np.asarray(centers)[:,None],3,1)
    elif representation == 'triplet': indices = np.clip(np.asarray(centers)[:,None]+np.array([-1,0,1]),0,n-1)
    else: raise ValueError('Choose single or triplet')
    pixels = torch.from_numpy(image[indices].astype(np.float32))
    # Average support across the input channels before patch weighting.
    valid = torch.from_numpy(support[indices].mean(1,keepdims=True).astype(np.float32))
    return pixels, valid


def protocol_codes(record):
    plane = {'Sagittal':0,'Coronal':1,'Axial':2}.get(str(record.get('Anatomical_Plane')),3)
    def binary(v):
        if pd.isna(v): return 2
        return int(float(v)) if str(v) in ['0','1','0.0','1.0'] else 2
    return np.array([plane,binary(record.get('Fluid_Sensitive')),binary(record.get('Fat_Suppression'))],np.int64)


def feature_name(record):
    return fingerprint([record[ID],record[SID]]) + '.npz'


def check_features(path, record, feature_id):
    with np.load(path, allow_pickle=False) as z:
        if str(z['feature_id'].item()) != feature_id or str(z['study'].item()) != record[ID] or str(z['series'].item()) != record[SID]:
            raise ValueError('Feature cache identity mismatch')
        f = z['features']; positions = z['positions']; protocol = z['protocol']; spacing = z['spacing']
        if f.ndim != 2 or f.shape[1] != 768 or not 1 <= len(f) <= 64 or not np.isfinite(f).all():
            raise ValueError('Invalid cached features')
        if positions.shape != (len(f),) or not np.isfinite(positions).all(): raise ValueError('Bad feature positions')
        if protocol.shape != (3,) or not np.array_equal(protocol,protocol_codes(record)): raise ValueError('Protocol mismatch')
        if spacing.shape != (3,) or not np.isfinite(spacing).all() or (spacing<=0).any(): raise ValueError('Bad spacing')
    return path


def extract_features(series, encoder, identity, feature_identity, work, resume_roots, cfg, budget):
    feature_id = fingerprint(feature_identity)
    folder = Path(work)/'features'/feature_id; folder.mkdir(parents=True, exist_ok=True)
    atomic_json(folder/'identity.json',feature_identity)
    sources = [folder] + [Path(p)/'features'/feature_id for p in resume_roots]
    device = next(encoder.parameters()).device
    records, completed = series.to_dict('records'), []
    for i, r in enumerate(records):
        name = feature_name(r)
        existing = next((p/name for p in sources if (p/name).is_file()),None)
        if existing is not None:
            check_features(existing,r,feature_id)
            # Carry prior features into this output so the next resume needs only this cumulative run.
            target = folder/name
            if existing.resolve() != target.resolve():
                if shutil.disk_usage(work).free < cfg['min_free_gb']*1e9 + existing.stat().st_size:
                    raise OSError('Insufficient disk to retain resumed features')
                temp = target.with_suffix('.tmp'); shutil.copy2(existing,temp); temp.replace(target)
            completed.append(str(target)); continue
        if budget.expired():
            atomic_json(Path(work)/'status.json',dict(status='FEATURES_PARTIAL',completed=len(completed),total=len(records),feature_id=feature_id))
            print('Session budget reached. Save this PRIVATE output and attach it on the next run.'); return None
        if shutil.disk_usage(work).free < cfg['min_free_gb']*1e9: raise OSError('Insufficient free disk for features')
        image,support,meta = read_native(r,identity['run_id'])
        centers = np.arange(int(meta['selected_count']))  # all real candidates; training samples later
        chunks=[]
        for j in range(0,len(centers),cfg['image_batch']):
            x,v = make_slice_inputs(image,support,centers[j:j+cfg['image_batch']],cfg['representation'])
            ctx = torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else nullcontext()
            with torch.inference_mode(),ctx:
                f = encoder(x.to(device),v.to(device)).float().cpu().numpy()
            if not np.isfinite(f).all(): raise ValueError('Encoder produced nonfinite features')
            chunks.append(f.astype(np.float16))
        target=folder/name; tmp=target.with_suffix('.tmp')
        with tmp.open('wb') as out:
            np.savez_compressed(out,feature_id=np.array(feature_id),study=np.array(r[ID]),series=np.array(r[SID]),
                features=np.concatenate(chunks),positions=np.asarray(meta['slice_positions_mm'],np.float32),
                protocol=protocol_codes(r),spacing=np.asarray(meta['spacing_summary_drc_mm'],np.float32))
        tmp.replace(target); check_features(target,r,feature_id); completed.append(str(target))
        if i%25==0 or i+1==len(records): print(f'Features {i+1}/{len(records)}; elapsed {(time.monotonic()-budget.start)/60:.1f} min',flush=True)
    result=series.copy(); result['feature_path']=completed
    result.to_csv(Path(work)/'feature_series.csv',index=False)
    return result


### Study batches and attention architecture

In [ ]:
def sample_centers(n,count,rng=None):
    if n<=count: return np.arange(n)
    if rng is None: return np.linspace(0,n-1,count).round().astype(int)
    bins=np.array_split(np.arange(n),count)
    return np.asarray([rng.choice(b) for b in bins])


class FeatureStudies:
    """Compact arrays in RAM; raw MRI volumes are never stacked across studies."""
    def __init__(self,series,labels,max_ram_gb=5):
        self.labels=labels.set_index(ID).loc[sorted(labels[ID])].reset_index()
        self.ids=self.labels[ID].tolist(); self.items=[]; estimated=0
        by={uid:g for uid,g in series.groupby(ID)}
        for uid in self.ids:
            features=[];protocol=[];spacing=[]
            for _,r in by[uid].sort_values(SID).iterrows():
                with np.load(r.feature_path,allow_pickle=False) as z:
                    f=z['features'].copy();p=z['protocol'].copy();s=z['spacing'].copy()
                estimated+=f.nbytes
                if estimated>max_ram_gb*1e9: raise MemoryError('Feature RAM budget exceeded; increase max_feature_ram_gb if available')
                features.append(f);protocol.append(p);spacing.append(s)
            if not features: raise ValueError('Empty study')
            self.items.append((features,np.stack(protocol),np.stack(spacing)))
        self.y=np.nan_to_num(self.labels[TARGETS].to_numpy(np.float32),nan=0)
        self.mask=self.labels[[t+'__mask' for t in TARGETS]].to_numpy(bool)
        self.gold=self.labels[[t+'__gold' for t in TARGETS]].to_numpy(bool)
        self.weights=self.labels[[t+'__weight' for t in TARGETS]].to_numpy(np.float32)
        self.groups=self.labels.patient_group.to_numpy(str);self.folds=self.labels.fold.to_numpy(int)
        print(f'Loaded {len(self.ids)} studies; feature payload {estimated/1e9:.2f} GB')

    def split(self,fold):
        val=np.flatnonzero((self.folds==fold)&self.gold.any(1))
        held=set(self.groups[self.folds==fold])
        train=np.flatnonzero(~np.isin(self.groups,list(held)) & (self.mask & (self.weights>0)).any(1))
        if not len(val) or not len(train): raise ValueError(f'Empty train or verified validation for fold {fold}')
        if set(self.groups[train])&set(self.groups[val]): raise ValueError('Group leakage')
        return train,val

    def scaler(self,indices):
        x=np.concatenate([np.log(self.items[i][2]) for i in indices])
        return dict(mean=x.mean(0).tolist(),std=np.maximum(x.std(0),1e-3).tolist(),transform='log_mm')

    def batch(self,indices,scaler,centers,device,rng=None):
        count=max(len(self.items[i][0]) for i in indices);b=len(indices)
        x=np.zeros((b,count,1536),np.float32);p=np.zeros((b,count,3),np.int64)
        s=np.zeros((b,count,3),np.float32);m=np.zeros((b,count),bool)
        for j,i in enumerate(indices):
            fs,protocol,spacing=self.items[i];n=len(fs)
            for k,f in enumerate(fs):
                selected=f[sample_centers(len(f),centers,rng)].astype(np.float32)
                x[j,k]=np.concatenate([selected.mean(0),selected.max(0)])
            p[j,:n]=protocol;s[j,:n]=(np.log(spacing)-np.asarray(scaler['mean']))/np.asarray(scaler['std']);m[j,:n]=True
        return tuple(torch.as_tensor(v,device=device) for v in [x,p,s,m])


class FindingAttention(nn.Module):
    def __init__(self,hidden=256,attention_dropout=.1,dropout=.15):
        super().__init__();self.hidden=hidden
        self.proj=nn.Sequential(nn.LayerNorm(1536),nn.Linear(1536,hidden),nn.GELU())
        self.plane=nn.Embedding(4,hidden);self.fluid=nn.Embedding(3,hidden);self.fat=nn.Embedding(3,hidden)
        self.spacing=nn.Linear(3,hidden)
        self.query=nn.Parameter(torch.randn(12,hidden)*.02)
        self.attn=nn.MultiheadAttention(hidden,4,dropout=attention_dropout,batch_first=True)
        self.fuse=nn.Sequential(nn.LayerNorm(5*hidden),nn.Linear(5*hidden,hidden),nn.GELU(),nn.Dropout(dropout))
        self.weight=nn.Parameter(torch.randn(12,hidden)*.03);self.bias=nn.Parameter(torch.zeros(12))

    def forward(self,x,protocol,spacing,mask,series_dropout=0.):
        mask=mask.bool()
        if not mask.any(1).all(): raise ValueError('Cannot predict an all-empty study')
        if self.training and series_dropout:
            keep=(torch.rand_like(mask.float())>=series_dropout)&mask
            empty=~keep.any(1); first=mask.float().argmax(1)
            keep[torch.arange(len(mask),device=mask.device)[empty],first[empty]]=True;mask=keep
        h=self.proj(x)+self.plane(protocol[:,:,0])+self.fluid(protocol[:,:,1])+self.fat(protocol[:,:,2])+self.spacing(spacing)
        h=h.masked_fill(~mask[:,:,None],0)
        mean=h.sum(1,keepdim=True)/mask.sum(1)[:,None,None]
        q=self.query[None].expand(len(h),-1,-1)
        a,_=self.attn(q,h,h,key_padding_mask=~mask,need_weights=False)
        g=mean.expand_as(a)
        f=self.fuse(torch.cat([a,q,g,(a-g).abs(),a*g],-1))
        return (f*self.weight[None]).sum(-1)+self.bias


### Loss, metrics and validation

In [ ]:
def masked_bce(logits,targets,mask,weights):
    use=mask.bool() & (weights>0)
    clean=torch.where(use,targets,torch.zeros_like(targets))
    w=torch.where(use,weights,torch.zeros_like(weights))
    losses=F.binary_cross_entropy_with_logits(logits,clean,reduction='none')
    denom=w.sum(0);active=denom>0
    if not active.any(): return logits.sum()*0
    return ((losses*w).sum(0)[active]/denom[active]).mean()


def ranking_loss(logits,y,gold):
    result=[]
    for t in range(12):
        pos=logits[gold[:,t] & (y[:,t]==1),t];neg=logits[gold[:,t] & (y[:,t]==0),t]
        if len(pos) and len(neg): result.append(F.softplus(-(pos[:,None]-neg[None,:])).mean())
    return torch.stack(result).mean() if result else logits.sum()*0


def ema_update(ema,model,decay):
    with torch.no_grad():
        for ep,p in zip(ema.parameters(),model.parameters()): ep.mul_(decay).add_(p,alpha=1-decay)
        for eb,b in zip(ema.buffers(),model.buffers()): eb.copy_(b)


def evaluate_metrics(y,p,gold):
    rows=[]
    for t,name in enumerate(TARGETS):
        v=gold[:,t];truth=y[v,t];score=p[v,t];positive=int((truth==1).sum());negative=int((truth==0).sum())
        auc=float(roc_auc_score(truth,score)) if positive and negative else None
        ap=float(average_precision_score(truth,score)) if positive and negative else None
        rows.append(dict(target=name,n=int(v.sum()),positive=positive,negative=negative,auroc=auc,average_precision=ap))
    aucs=[r['auroc'] for r in rows if r['auroc'] is not None]
    return dict(macro_auroc=float(np.mean(aucs)) if aucs else None,targets=rows,
                note='Average precision uses sklearn; it is not trapezoidal PR-AUC.')


@torch.inference_mode()
def predict_head(model,data,indices,scaler,cfg,device):
    model.eval();logits=[]
    for start in range(0,len(indices),cfg['batch_size']):
        ix=indices[start:start+cfg['batch_size']]
        logits.append(model(*data.batch(ix,scaler,cfg['centers'],device)).cpu())
    z=torch.cat(logits)
    if not torch.isfinite(z).all(): raise ValueError('Nonfinite validation logits')
    loss=masked_bce(z,torch.from_numpy(data.y[indices]),torch.from_numpy(data.gold[indices]),torch.ones_like(z))
    return float(loss),z.sigmoid().numpy()


### Fold training and atomic checkpoints

In [ ]:
def make_epoch_pool(data,train,cfg,rng,refine=False):
    gold=train[data.gold[train].any(1)]
    weak=train[~data.gold[train].any(1)]
    if refine or not len(weak):
        return rng.permutation(gold) if len(gold) else rng.permutation(train)
    rng.shuffle(weak)
    if not len(gold): return weak
    nw=max(1,round(cfg['batch_size']*(1-cfg['verified_fraction'])))
    chunks=[]
    for start in range(0,len(weak),nw):
        w=weak[start:start+nw]
        ng=max(1,round(len(w)*cfg['verified_fraction']/(1-cfg['verified_fraction'])))
        chunk=np.concatenate([w,rng.choice(gold,ng,replace=True)]);rng.shuffle(chunk);chunks.append(chunk)
    return np.concatenate(chunks)


def cpu_state(model): return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}


def train_fold(data,fold,cfg,work,resume_roots,training_id,budget,device):
    train,val=data.split(fold);scaler=data.scaler(train)
    seed_all(cfg['seed']+fold)
    model=FindingAttention(cfg['hidden']).to(device);ema=copy.deepcopy(model).eval()
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
    folder=Path(work)/'folds';folder.mkdir(exist_ok=True,parents=True)
    name=f'fold_{fold}.pt';target=folder/name
    candidates=[target]+[Path(r)/'folds'/name for r in resume_roots]
    previous=next((p for p in candidates if p.exists()),None)
    history=[];best_loss=float('inf');best_state=None;best_kind=None;best_epoch=None;bad=0;start_epoch=0;phase='mixed'
    if previous:
        saved=torch.load(previous,map_location='cpu',weights_only=True)
        if saved['training_id']!=training_id: raise ValueError(f'Checkpoint configuration mismatch: {previous}')
        if saved['complete']:
            model.load_state_dict(saved['best']);loss,pred=predict_head(model,data,val,saved['scaler'],cfg,device)
            if previous!=target: shutil.copy2(previous,target)
            return saved,pred,val
        model.load_state_dict(saved['model']);ema.load_state_dict(saved['ema']);optimizer.load_state_dict(saved['optimizer'])
        history=saved['history'];best_loss=saved['best_loss'];best_state=saved['best'];best_kind=saved['best_kind'];best_epoch=saved['best_epoch']
        bad=saved['bad'];start_epoch=saved['next_epoch'];phase=saved['phase']
    stages=['mixed']+(['refine'] if cfg['refine_epochs'] else [])
    start_stage=stages.index(phase)
    saved=None
    for stage in stages[start_stage:]:
        if stage!=phase:
            model.load_state_dict(best_state);ema.load_state_dict(best_state)
            optimizer=torch.optim.AdamW(model.parameters(),lr=cfg['refine_lr'],weight_decay=cfg['weight_decay'])
            start_epoch=0;bad=0;phase=stage
        epochs=cfg['epochs'] if stage=='mixed' else cfg['refine_epochs']
        for epoch in range(start_epoch,epochs):
            if bad >= cfg['patience']: break
            if budget.expired(): return None,None,None
            seed_all(cfg['seed']+fold*10000+epoch+(1000 if stage=='refine' else 0))
            rng=np.random.default_rng(cfg['seed']+fold*10000+epoch+(1000 if stage=='refine' else 0))
            pool=make_epoch_pool(data,train,cfg,rng,stage=='refine');model.train();total=0.;steps=0
            for start in range(0,len(pool),cfg['batch_size']):
                ix=pool[start:start+cfg['batch_size']]
                if budget.expired(45): return None,None,None  # Previous epoch checkpoint remains resumable.
                y=torch.as_tensor(data.y[ix],device=device);gold=torch.as_tensor(data.gold[ix],device=device)
                m=gold if stage=='refine' else torch.as_tensor(data.mask[ix],device=device)
                w=torch.ones_like(y) if stage=='refine' else torch.as_tensor(data.weights[ix],device=device)
                optimizer.zero_grad(set_to_none=True)
                z=model(*data.batch(ix,scaler,cfg['centers'],device,rng),series_dropout=cfg['series_dropout'])
                loss=masked_bce(z,y,m,w)+cfg['rank_lambda']*ranking_loss(z,y,gold)
                if not torch.isfinite(loss): raise ValueError('Nonfinite training loss')
                loss.backward();nn.utils.clip_grad_norm_(model.parameters(),2.);optimizer.step();ema_update(ema,model,cfg['ema_decay'])
                total+=float(loss.detach());steps+=1
            current_loss,current_pred=predict_head(model,data,val,scaler,cfg,device)
            ema_loss,ema_pred=predict_head(ema,data,val,scaler,cfg,device)
            use_ema=ema_loss<current_loss;score=ema_loss if use_ema else current_loss
            pred=ema_pred if use_ema else current_pred
            if score<best_loss-1e-6:
                best_loss=score;best_state=cpu_state(ema if use_ema else model);best_kind='ema' if use_ema else 'current';best_epoch=f'{stage}:{epoch+1}';bad=0
            else: bad+=1
            metric=evaluate_metrics(data.y[val],pred,data.gold[val])
            row=dict(phase=stage,epoch=epoch+1,train_loss=total/max(steps,1),validation_loss=score,
                     macro_auroc=metric['macro_auroc'],best_loss=best_loss)
            history.append(row);print(f'Fold {fold}: {row}',flush=True)
            saved=dict(training_id=training_id,fold=int(fold),complete=False,model=cpu_state(model),ema=cpu_state(ema),
                       optimizer=optimizer.state_dict(),best=best_state,best_loss=best_loss,best_kind=best_kind,best_epoch=best_epoch,
                       history=history,bad=bad,next_epoch=epoch+1,phase=stage,scaler=scaler)
            atomic_torch(target,saved)
            if bad>=cfg['patience']: break
        start_epoch=0
    if saved is None and previous: saved=torch.load(previous,map_location='cpu',weights_only=True)
    if saved is None: raise ValueError('No training epoch completed')
    saved['complete']=True;atomic_torch(target,saved)
    model.load_state_dict(best_state);loss,pred=predict_head(model,data,val,scaler,cfg,device)
    return saved,pred,val


### Model-package export

In [ ]:
def export_package(encoder,data,results,cfg,identity,feature_identity,training_id,work,runtime_source,preprocessing_source,preprocessing_config):
    package=Path(work)/'model_package';package.mkdir(exist_ok=True)
    encoder.export(package/'encoder')
    folds=[];oof=np.full_like(data.y,np.nan);rows=[]
    for saved,pred,val in results:
        fold=saved['fold'];name=f'head_fold_{fold}.pt'
        atomic_torch(package/name,dict(state_dict=saved['best'],scaler=saved['scaler'],fold=fold,hidden=cfg['hidden']))
        oof[val]=pred;folds.append(dict(fold=fold,file=name,sha256=file_hash(package/name),best_epoch=saved['best_epoch'],
                                      weight_type=saved['best_kind'],validation_loss=saved['best_loss']))
        rows.append(dict(fold=fold,**evaluate_metrics(data.y[val],pred,data.gold[val])))
    gold_rows=data.gold.any(1);covered=np.isfinite(oof).all(1)&gold_rows
    frame=pd.DataFrame(oof,columns=TARGETS);frame.insert(0,ID,data.ids);frame['fold']=data.folds
    frame.to_csv(Path(work)/'oof_private.csv',index=False)
    metrics=dict(per_fold=rows,pooled=evaluate_metrics(data.y[covered],oof[covered],data.gold[covered]),
                 covered_verified=int(covered.sum()),eligible_verified=int(gold_rows.sum()),
                 complete_oof=bool(covered[gold_rows].all()))
    atomic_json(Path(work)/'validation_private.json',metrics)
    # Synthetic inputs only: exported smoke reference contains no patient image/features/IDs.
    seed_all(991);test=dict(x=torch.randn(2,3,1536),protocol=torch.zeros(2,3,3,dtype=torch.long),
                          spacing=torch.zeros(2,3,3),mask=torch.tensor([[True,True,False],[True,False,False]]))
    check=FindingAttention(cfg['hidden']).eval();check.load_state_dict(results[0][0]['best'])
    with torch.no_grad():test['expected']=check(test['x'],test['protocol'],test['spacing'],test['mask'])
    atomic_torch(package/'synthetic_head_smoke.pt',test)
    (package/'knee_runtime.py').write_text(runtime_source)
    (package/'native_preprocessing.py').write_text(preprocessing_source)
    atomic_json(package/'preprocessing_config.json',preprocessing_config)
    import importlib.metadata
    versions={}
    for name in ['torch','numpy','pandas','scikit-learn','transformers','timm','safetensors','scipy','pydicom']:
        try:versions[name]=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:pass
    (package/'requirements.txt').write_text('\n'.join(f'{k}=={v}' for k,v in versions.items())+'\n')
    manifest=dict(schema_version=1,runtime_version=RUNTIME_VERSION,training_id=training_id,targets=TARGETS,
                  dataset_identity=identity,feature_identity=feature_identity,configuration=cfg,folds=folds,
                  environment=versions,grouping='study-separated' if np.array_equal(data.groups,np.asarray(data.ids)) else 'provided group-separated; patient identity must be audited',
                  complete_oof=metrics['complete_oof'],source='generic public DINOv2; no task-adapted encoder',
                  model_scope='frozen encoder plus attention head; encoder fine-tuning not implemented',
                  inference=dict(centers=cfg['centers'],sampling='rounded linspace over valid slices',ensemble='mean sigmoid',
                                 input_shape=[64,320,320],padding=[1,1,1,1],feature_pool='CLS + support-weighted patch mean; series mean+max',
                                 missing_study_policy='error; no silently substituted predictions'),
                  privacy='Package excludes source scans, reports, cached features, labels, IDs and OOF rows. Do not publish the whole training output.',
                  validation_limitations='Small verified set already informed label development; no independent test claim.')
    (package/'README.md').write_text('Frozen DINOv2 Small knee MRI model package. See manifest.json for the input contract.\n'
        'Heads use fold-specific log-spacing scalers. Generic encoder is shared. Average sigmoid outputs.\n'
        'Reference preparation code is native_preprocessing.py; no test-label or report input.\n'
        'This package is not a submission notebook and contains no submission.csv.\n'
        'Review source competition and encoder licenses before distribution; no automatic upload occurs.\n')
    manifest['files']={str(p.relative_to(package)):file_hash(p) for p in package.rglob('*') if p.is_file() and p.name!='manifest.json'}
    atomic_json(package/'manifest.json',manifest)
    return package,metrics


## 3. Preflight and dataset audit
All active shards are required; legacy and partial shards are rejected. The original verified labels are cross-checked against `train.csv`. Patient-group separation is enforced where provided; a study-ID fallback is reported honestly.

In [ ]:
import inspect, importlib.metadata, platform
if MODE not in {'all','features','train'}: raise ValueError('Invalid MODE')
if not FOLDS or len(set(FOLDS)) != len(FOLDS): raise ValueError('Choose unique folds')
if not (0 < CFG['verified_fraction'] < 1): raise ValueError('verified_fraction must lie between 0 and 1')
if CFG['hidden'] % 4 or min(CFG['epochs'], CFG['batch_size'], CFG['centers'], CFG['image_batch']) < 1:
    raise ValueError('Invalid model/training dimensions')
if not 0 <= CFG['series_dropout'] < 1 or CFG['refine_epochs'] < 0 or CFG['rank_lambda'] < 0:
    raise ValueError('Invalid regularization/refinement configuration')
if not 0 < MAX_HOURS <= 8.5: raise ValueError('Use a bounded session budget of at most 8.5 hours')
if REQUIRE_GPU and not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU accelerator')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BUDGET = SessionBudget(MAX_HOURS)
seed_all(CFG['seed'])
ROOT = discover_bundle(DATASET_ROOT)
SERIES, LABELS, ALL_LABELS, INDEX = load_bundle(ROOT)
EXPECTED_PREPROCESSING_IMPLEMENTATION = 'ef1a2effbba938df32d512f010e3786904efc0cda3d04c95b074e29265018b5d'
if INDEX['preprocessing'].get('implementation') != EXPECTED_PREPROCESSING_IMPLEMENTATION:
    raise ValueError('Preprocessing implementation changed; update the embedded inference transforms before training')
if COMPETITION_ROOT is None:
    candidates = [p.parent for p in find_input_files('/kaggle/input','train.csv')
                  if (p.parent/'train_series.csv').is_file()]
    COMPETITION_ROOT = select_one(candidates, 'COMPETITION_ROOT')
OFFICIAL = pd.read_csv(Path(COMPETITION_ROOT)/'train.csv', dtype={ID:str})
if not OFFICIAL[ID].is_unique or set(OFFICIAL[ID]) != set(ALL_LABELS[ID]):
    raise ValueError('Official training IDs do not match the custom dataset')
original = OFFICIAL.set_index(ID).loc[ALL_LABELS[ID]]
for target in TARGETS:
    gold = original[target].notna().to_numpy()
    if not np.array_equal(gold, ALL_LABELS[target+'__gold'].to_numpy(bool)):
        raise ValueError('Verified-label mask drift: '+target)
    if not np.array_equal(original[target].to_numpy()[gold], ALL_LABELS[target].to_numpy()[gold]):
        raise ValueError('Verified-label value drift: '+target)
available_folds = sorted(LABELS.loc[LABELS.has_gold.eq(1),'fold'].unique().tolist())
if not set(FOLDS).issubset(available_folds): raise ValueError(f'Eligible verified folds are {available_folds}')
print('Device:', DEVICE)
print('Usable studies:', len(LABELS), '| series:', len(SERIES), '| excluded studies:', len(ALL_LABELS)-len(LABELS))
print('Grouping:', 'STUDY ONLY — no patient-separation guarantee' if
      ALL_LABELS.patient_group.eq(ALL_LABELS[ID]).all() else 'provided groups; verify patient provenance')
display(audit_labels(LABELS))
ENCODER_PATH, BACKEND = discover_encoder(ENCODER_PATH, ENCODER_BACKEND)
ENCODER_HASH = encoder_source_hash(ENCODER_PATH)
print('Offline encoder:', ENCODER_PATH, '| backend:', BACKEND)
ENCODER = FrozenEncoder(ENCODER_PATH, BACKEND).to(DEVICE).eval()


## 4. Identity and restart discovery
Resume folders are private outputs from this notebook. Feature reuse checks encoder, preprocessing, code, representation and numerical-library identity. Head checkpoints additionally require the same complete training configuration.

In [ ]:
# Capture executed definitions so the exported code is exactly the code used here.
RUNTIME_HEADER = '"""Offline RSNA frozen-DINO training runtime. Embedded verbatim in the notebook."""\nimport os\nos.environ[\'HF_HUB_OFFLINE\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nimport copy, gc, hashlib, io, json, math, random, shutil, tarfile, time\nfrom contextlib import nullcontext\nfrom pathlib import Path, PurePosixPath\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import roc_auc_score, average_precision_score\n\nID, SID = \'StudyInstanceUID\', \'SeriesInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nRUNTIME_VERSION = \'rsna-frozen-dino-v1\'\n\n\n'
RUNTIME_NAMES = ['fingerprint', 'file_hash', 'atomic_json', 'atomic_torch', 'seed_all', 'SessionBudget', 'select_one', 'find_input_files', 'discover_bundle', 'safe_member', 'load_bundle', 'validate_labels', 'audit_labels', 'read_native', 'discover_encoder', 'encoder_source_hash', 'FrozenEncoder', 'make_slice_inputs', 'protocol_codes', 'feature_name', 'check_features', 'extract_features', 'sample_centers', 'FeatureStudies', 'FindingAttention', 'masked_bce', 'ranking_loss', 'ema_update', 'evaluate_metrics', 'predict_head', 'make_epoch_pool', 'cpu_state', 'train_fold', 'export_package']
def definition_source(name):
    obj = globals()[name]
    if not inspect.isclass(obj):
        return inspect.getsource(obj)
    # inspect.getsource(Class) fails in Jupyter's __main__; recover its defining cell.
    import ast, linecache
    text = ''.join(linecache.getlines(obj.__init__.__code__.co_filename))
    node = next(n for n in ast.parse(text).body if isinstance(n, ast.ClassDef) and n.name == name)
    return ast.get_source_segment(text, node) + chr(10)
RUNTIME_SOURCE = RUNTIME_HEADER + '\n\n'.join(definition_source(name) for name in RUNTIME_NAMES)
import base64, zlib
PREPROCESSING_SOURCE = zlib.decompress(base64.b64decode('eJzNXHtv20iS/z+fos+LW0lZSrGdnewlGS4mO0l2A0weiDODwwmCQJMtiRuK1LBJ25pc9rNfvfpBirKdAe5wg0EsUd3Vj3r9qrqaq7raql3SbIr8UuXbXVU36gN8fSCfN4nBnyL1T1OVkWryrY5UZSJlNm2TF5HKG103VVUY26Nst7u9Sowqdw9WSN2k+W4/K7N8m6y1HSNZrfJSL5s6Kc2qqreRWietMXlSLld5ATQtud0+y9Nqy6Tky2yX3+jCOFq7XbFfbqssKfJmvyza5sGbl/Hoommz/ZvSNEmZ6p/fvBw9uKDHus616Tx/8Z+vLlSsvjxQ8N/oIlnnTZMUo2ewhFlS18l+PJ+fRafR6SJSc/gTnfGHKTxcwMdVUSXNJOLuP1Z1VfZ7u04BmcHeL27yo33hD/dlIkHXrw8yvVJXsP4safQyKbOlvoG9TZvlWldb3dT78UYnma6Bc/pmp9NGZ0uDGxR+p52J1K5ISuByulpPntGsTk5O6O8vMoBRL9/8+P6tYpLKDgFdNjr9bBQwVBkQKliKSqvS5KbRZep/J2KmyFM9hWZpXq7DZuo6bzZ5iY9W+bqtdaaaqtA1sgtmB4tTsjijmo1Wu83e5CkNVdVZXsIE1WUC1NQ2aer8ZkbD0T8fkjqBqcI28MLwP9kXNS5g/Mkz9RP8UdXKypqCBSdGN64dLi5RBiZdaMVbNnPEunurxqapgeQrear6IknEhG15VQ7RoQEOCfWF+CglYuZB/wR2qdrSrkkDL/aRl+HICuTEEwSxUGPYGtyrD/lOF6DHjlc0uMJf4W9S7/GHJslLZLFnoidGHz7qpq3LgCXY/5m6EAkChiXIBJWXadFmSIpkRwG3dQ3WqM7XOdimkOcsJtdJjSPLeFaK85WsuawaoKlQ+/3YdZIbjZLe6ld1XdXj1cnP5eeyui651zP1hf5+PZlYaoUurXZN1Pfq/BZiJ+807n4DfRIQs+a6copCazJAlXqv8hp+j63UzU8XXor/oPJqp2CtSanISKC4PmFjAMud7qq8JDt8SXKNMpIldQYbmRHH0RDjLtqNFS3/g5qqT6BPPPRjlKYWRQ8Gmz97vAB2v8xrTZwFtoICaFYG1EG27XV1PeuQKpIDSo+f3YdSWhXtVqQYFxuTQTRsEmmCszfY8D2oAQgYUgKvhZ+tUfTbZU3MAI0P6Ecu+HfXUVZwNnP2DqQkJx5VfjgSMEt65+yKlQmY9Mxskp1W/xar8ZNoAn1J4GAOuVkhQY37MZklRTGe3CYyb0rS63BwL3syg2Cs8+GxpKGMhy22eemequ9jdXqfSZDntaNaYQVuRch82GIRlkgJr+08x452cmnGMDEwG0mxnpXg+8fQfwISczZRf1VnevrYtUVTe9gcRrq1OZBTPyhqFTS4dY8DOVJXIJkVKE5Ss4kAiLGpcOCksAvmbyxRaV0ZGpO2gBmz21RNBT+D1bWShk/QS+bpmxIEZVdrHs5xkvuITfpy8vb9u/c//uPj+7evzk4iFXw9P/l660rKYq+2MN90A4hJq7cf3yjT7hAo6czOn6UFhAUMhUzwY3VtJsA29+BHUkEzkRWBmuJ0DfSaM1dNtZNvojHnM/UGloYKg8ph4UjHtye0XQqEUAwerhb1nh0d2z7g4wafi/Hzy0WBhy3dzPqOdIKi33O+CEK4cd9Z9lrTz36Q4X1lxEOUH4lTBq+7zQ04nHQjCmmVHzZxjfzF4YF578gUv1+9RithgJ1nE5rC2V2Dvio3OOXs0bYtmnyF3QV6tWXI1FvGvki2u0KbD7omY/cNg7+ryukabKUBnyHDdoci6dmEkrPxUoNDkJjdNcxbmFYGLn+rQUpQwAT+8SZ3RxSjBtKTFpXRMN5RLwD2B0AAYI4YLMCfQTfx4+nkfrMJTO03zqfrUYz98Hsn0jG3t82EJf24lUF2kIW557ieEDI2oBQM6zGmNQ4zCMV0mY0DRyss+iBNel7aE0NrYruL2r7/EOqsmC7BW4DIx9iFBY2e4LfbLOPLdgcWBw0SUIZVMWmHuEILF8zfPfaG2q3WO90QAkbq8aAH9pTu9vlvc4MRxqMScTX2FshraZw4mPJ4pi4wCO7+bjhMQuCFC7YYc53s2MQSdJaF1msDBPz0wHeyf4vU57zM4hPYqMtCnxw6Avd5TvTYLWS5bGzYwNGkJjgNHjzLV6ux68EjZL/BbyQeKEcIWpNyjF0CEYA2iFgsjKFfAXqjs7+XCODGTjvI28/VgyscPrkZI6agKU9h3AmCCnx6Ojv9joLk+Qn8uHTRzclCPcR2t6KputbrtkhqGdsq+CN0KMZFOOY5CZHJQbwacOusiVWRBK681ibPWgIjfrenyrMTQgf1J2Z0Uq51R1Qn82eRAjuvec7wD3Mp3IEO8pLBwKDd5CY+m9D+4I6czm4Pe6oSMXJRgDV7BH4hX2EUKotvkvSzF+g/z9SP1XbXEopwUWp1WeS/tnmzJ8FeFcnaRXc0TwirdEYUEgrMAwnK9LrWmlBkUqcQb+CntMh3xFiBcj9QGDin0G6Bu3K6AEU+JW8pVqrUgAgpKEORw+agIHofF8n2MkvUjpI23zTYkYGEr1e5vl7KImFQH73yCv/KsgcMWLrNWcrwJ4uAGX1C444HWHWwLzeGGJfGeDY7W31VQlNRBg6TJMA5iYGfq5MeMTtMs4G92lQFNMVpjganOVrQCM/d1nJc/kW+fp31yJ+8g1legVykQIbFW6FfggAyey6di+QSZKwt0w1KezbzFJyEfYcSBua/btOGsOcKRVw9fnksg0T9+AFDfgI5SxLc8VyYigrEUdBDq89zzNZhNOKfnC0WHSZj5oPyHZ4rZEvjTnIjDrVZMh0x/Rt5B4rOKO55oof0FD6QQ4/p34gFKKZ/I5h1DBM/Ii5x92tkObWkzY7lWzAJ1OgloJT0MwT1JmYltHiUAgvEpNjqk20EkPSUHOYp7wzwidKZJTF7ua7zbLy+JSsJ0BYzyMZ5OWZI/huLNPwPXiy/YisJYLYU44rRCgQi+brEdDaCXPgrJiQDtpEN4hTj+UtJWCuXsGbiJB/o69gAKYjgtskOAhcw1NVal7pqbboTBi7B5xbJvmobA9OCtgkIas7pb5ulyX9D/Nk2MLjCpRPog+8Qjl7mMHq9R0GtK0GndyQ3faruIJsWJOkknQoDk47bfO7/93wiIBeN3vE3HSZj7kouNoAB9DM1tonCarUCMClKBfJVV7uJag1uBshkVl2jpdDJ1ua4tMGIyg1kpZASRhe6gP0QyQWxm/JmJDfaOD6CZaza9UZ+8i6dZLUO2Ox4zN6HacYKjClNdUTe40xSLCBH6Jmk1aO+4+bn4rZPAwtEUfsntL2c38EGaqfrbWvDH9paUBJwqDlgIJJwiHhJL4QFoZ/2zoRTQ5sqZywoWQPcVKSP8f18TH4vUuMztAGB06ITBNRMbOaOmGa7usrAaI/nU+iAZyK13umkic974VRm04s4MG4NbhWOijjHHa3QAJNOR5NWdQc8mHY7Dqg97MME2P9Jl4Ss2MYyYyIZBVPqtX9wMGHEVEKlgzBunqkbsP8TcCKdLI5bEIgGPQPRwHmFXIYw4cekSFvEjv7EJIi6x2AEgHMTzCVTPgYN5SrX4MHhCToBNX79/peJJPzqEnPLwdgIDMaHnHpIPC45XYdcRWvMM19YvANt/+lXP7bEfxBpnn2awGe/Q9SHZ8sT2DW7cUBjUMoBVb5E27hF69KJqsEC7Mi6FICw0Q3wWQWsXcBBWxQKVq6uN4DDYbIa4u8rbJQYtH6KjFQv08zyQ8hHHi6329HC4Wr4ZQaaMx7BKEscYbmqrsA8vk4KowNh9hRRKEwnUy0cYwg+AZ2X8cAejha04b1QXZAPY+u73dqNPRaA/zCZ7y2f/I6uzlhfxXvqAZQBg5p+xo0Um2Sqtk51p4Wn33LQw7vPDFTmGskz3AfL81wsNWUzi1w8M02crecMIpkB84kHJWjOZx0U19Fp2VdqIEuLAwu6y8srsaAoiUQiaDtHHQBwz98kDD2DqAsC1AOS1xtd63GnB7AKvteYPxMSjkbU/yUSYh3ZfhLqdqrRo+Bu8naBZgd7hCcVjVNxgQSEgbao/KzcRAE1kQ0Mqe4jdc6JA+ZBbFtN7ep+wKmiJI/Po0NJxP6TgIDdM/42sGdunP6eyQ9+z4TEwJ7JL9bFd/bsL8HpjklWuiFoAZ4ewEVDXg4MYwu+QN+AGTYowIgMdrixWYWAD08G0rStec9qe0gF2phvwWmgZ8M4/GAnHgbRftCelTnCNTydeDsBdF2M7QIpfLpcIWQDlo4wvIde/3HbGeZIIie0YzTX66oFy34JwZYfAoKxf4dgLC9TwDtAAGf9SCY76tkSiVxuAVGC4jOdVpleCpgn5D3GepMeiP+pSjI60AywuqTcKUmwyjFW2SbmM1kcMi5rAFJgHXZJRofCfMpoIXxgI5iOrQ9RP/38CYEWmDNKbPuzOz405pVJaUFdIfcRC6LFfXzOxBkJJkVVrk2ewScGT0gcZ3gHJsfVE4DGWXygrVCvybXgDyB5fs609vsCWppVxFNZ4jy6OGMq00ZFwkgDnDYM/Ph8Cgazf3bMp8rJCjXc7ZtY5+d0pCRHuhj2kCpskivdjWEcyED8FRz581z8LHsT+hvAB52UtJGKS4HImKGLEUYjlzB/J5lRnsoAGkeZCjYTRah7MNVsEvbx7Lx41eh5MKjDhNv5S0ZnlMuU2qMs3YJ6ZCzErAsJnnxmhsuSlkTGKjD8RqVPmCW+NUPmQimuLJnyYQ94aK516qfdCbe/xT3SN5jTBB7yKrva8ByWBwtyFguktWzy1Z4O4JKcRDBA716raBAOCcisVRCpj3Ex5BNA0YFNzkptEkPxfQYqdEInIB94FrTAkwDPJA4bwXYdtPRrvHTtbO7ggPZHzO/8lG/z5gQUN8A6POs/gv9CxnxP2boEZgwu4r/5GSdw+Vl/WwEov8B6MpKLYzbDFNVOP6LYLdW7BqChLi0uAau6D82OxNuB4YgHKtZwXhAk4NmAafY7cndic/o4lg0ADYEaeYkHrpm1SZy5uCRMh4It0kDahBEocIDVZkoCwhVZoUUPrYj6Y7dcAX+y2Rk+hFriHkAEgYtAVhhSCzOUpnEQxQA0u2wh4lwX1SUmDEj5HCGF2VLys5coiMYdGZcKZbe2R8VKcEJoKo0AyQS2fVvV+6kGeJvygVMm8B/ilDxVDsqiy2/0en8Po31YI7ayZtsM2O3e0fY3ZTyCNIebKQfbjIh3KHiwGzA8MIPKU1wy6G5PUVQga5t8vZn0Db/nArOTQjawvSDXtWPMgKn1sdXh1ttwiw6i1WWbYcoAFiCeHTYOYqorbPjxBWzdLv8sdQDQhgGDhD9nfNwy8jLHNJdsIgEIPXpER4HEEVFtbhIkILz5tMUKkpFROkk3PKlIYEun2lDhJIs9lSxq8SKdtfnqBfLkecmiEZQvhLoVH4dGoSlrae7Uc04dF54cH4Vyo14G5A82F4VqQ/Omg7k6v2zJybSXiKKxzJFLaVlvUBKTuhMPucF8TEQbgAA3GB2xbUSm1nGt8/vEGbbcxhn2P9k8mzDhDvP8ZjGQJyH7/EvHfyewGCdqXPRGGQvPHqOuIYDARHwh2Uq2KL2yEznPtwKDGSjA5RDjYk49ss8p0Hh8mysfvSmDGYXz6IPozmEXKKWuSd7Aj+g6UHGDORFXC4vBEJZXn86+Q3MO7Z8+xY8TClrorDY3G2tjvUKzNaVxnf5L/sSNRIdVVYknpBihj+2aRfGCKY266aXvgCuwk6u9x1SptfeZtSVYUCWYzRYnUnRf1VvgIiK6TGPOnJSSDitDvoSeaO5WsAgK2mhF38e4vNvYQ2c/SdmA/2bmiD9xW3Uk1mE8AsQnNgFDJtSeVsihQ0+fQ5WXU5jI7390EEAd+s13lrDpeUqq1d7ADqKEdAs2rmBvc6zXt2GQZO0AiXKqCza+yadANiG9d5lu8TU2ErK6IMkiNsbsd0txeEfPLcQTl8Mppjvc7f1DFcRzPC/qMwttJ8xiOLjohGu90lNW1eDAg+ry5MDjw9G6GDXWs/UsUqOgcG+EUhk8OA/PPUAI1JirYABQkPIPYCB3juo7kpi7nj+TsbhfT0nVdPfkjiSg7y2Zl27vF0FXe4hB7X4v3AlOdSKSUi+kRbLG6wtNel+cw+JovZEUzA2GxmHL3grPngyJnVP3rKMqWV8KmXw49BGRlCYskZgESWk4EWNPPxTP7tkTZzPIRAhuyPmMcY65qAVbVFFLgd4HcI/iB5Agi/lMJ2yxpQxs1WBdaAwx8UwiyV+lluGOMAbRF1qp5tCmuVrYYTWDaDbQMMKyurYLRg9DpOQsFiDProCY/rKGCQKZXrFtHHfVtW9/Ykrh0eeBMPF1hfns3FVnD+WkBH/AWL/puvKbOf8XYzmX7/R4gKPP47b6Kk/U3+WulDLbqsKaQDbh7sARcEIr2e1xaNcnPuP+whEFW061RQd9sfooSMHAzgOSrMWnZJz/AhC7azYsjiZfbzHAn33Hp2zm15pLuCTFKcdpYoUePlTnro6I4ePpJDwgmY+c+gO0l2PIbeJLjjy7AE4gaqgwa927R2Y9MPWNMKLX8SgVCDAChwuMsGdF+F+my2qL5xODxIhrA7J9f/qBLlGiE8jYyUfh4BGqYAytUHAMRNefg5YwHiXF43C2VOv+ZOjM56NYj8DLuTyY84LDHl3McejRh5wF2wlXVh4kvQU68hc5AmCUEB9cArS86oMia52pwkUSUFwlc3Z0xyOMKplrMZ+q8RTFzA4Mfpy3/zvzAYbNnj592jvJ+S9gN2U7gwpv0Mk1n2zKoQRm7wKTx6g4IsvN0C2l20UVe62zJ2QdOC5XNi5P9wEz5v+SwbxFGoDd3Pbu8tXRO1e2yl2OIGrrT7jREc9x9mTifLfLPWEgZ4E2OUssuYvYGPXwM8fBxtfaBMGvAEcmwMUvCZbX0OH3FVZ5aV/wwOcKViNoKGEIJ6sgGtOYr5LTNemeELuuk72R+2v2JPIo9i2psB6gwaeqwWomF9ImV0leYCmuO1K9Nc1EM3S0eNaemPSt9a9tXvuk5W4QoN0FuDyiAbz00l9Kgx3/tWWXvHYJH9nuAQjzhneuoXX71cpUSdI1enuIMalWBuYODcVsdfkiEoaKIuM5ocYQkVqFNZJWHqVIFmUJpTDuZCs8eHkPI9bXIPYRni338iY4IxbRodRLkjatu+4XpiJYYLyHHq9bnE2jNXlfFB7D+NoJG2FRE5SINTkqN1Z2SKWzVBV1yozEyYrUx+68tJfdKdkps04dpm+CfLkxiOQ4G4MXe7TkWewQyDkRhSDbVwGYKBIOVii3jqU8tiqY2ZUwZcwjweSYxFioTiaI32h2fj42iS0sZ3Mhp3hy0We8gpgISyvl+vOdt57f1+lGc5qYGQUbP22qqUacB7Nm6oz2RHn4jjAYEkkHc4LBDkxUKXAn4EZJ3NykuB8wnxWabrlZvUtqYy+S0FG9r8iLOpcRiWSn7tH6AZi76KOkVu4Ovnme/SNKvrKJHoSym2HI5vPeslDMEHmb0bkLTTe37DkURBhBs85V5wtJygw07JQgfjooQcNaJeIGNvuddYb3jTCDMFBwFW615HumHNsguoqG6i4nQ4l0EQTGu/R2g6IIDjCdtc8sO56L/8VEL3pfdtwle56VO8vjU4qYaOpsjEwVTZjMMBocjx7i0ebIQ3D0/r0ENo+Eh9XvquY10nb1BdXB0ccXJv/1MOsJBvSVTZAWe5Jy3XmlgAnvCfKyrY0DSECRFTS82kv0KLfDwbZ8fPGWtUvIxGp+cG6Lml/tlpcatNSeHcSf6lZzgdrOJe4XPTv3Ss4BOmfI7mqNLZrFX0PVJFSBHJM5ET2s6Lrn6xqOm6le4PgRozOjw8vx/moE8J0mIchBatY3+JqOqiz1mh/7Kjosm8b2Uu9vL8y7KzxAuFuIeroYukOhvg/DNOjB9eyjBReyum9/OqT2kG9V8VkOlc6EdHptYzU96w4Dq/ej0Jf5s2fTs0VfFsOSuOHabov7pT7ElQH5whGXQZTdxWiJyN9SHuOKdYcK3HsxwUfN3AyOG4ES5luZSVaSMv8yAYBGAPGlMJqxSAiQulCMAwEcQmekMjjEPF9wUgH1we/iogcehrG455zEfzRq79DgL6hSOrXViHwLg18Lcrmfyq0s711xKZdtXmThVgcOzoQIgBraWVEjrsswY1lnpJyGrSO7nsNQL+CYsIYxxXH6cgD++6h3kYfsjpxUURU6vyul7mxMUWFBFx3ul3gCBJ4IZGEq8R6iSK5ld7iMbXTkk5jG33vwSX9/HOpzA0E6IHJiZzrujtwXhfGOJt6lICOe0MdabzDZB9xzTjFpsxxt+/qbjuEvODouBk/jQ3cUJM+Pve0FfVAmnXqvfPG9PYL45UDlnAMYQhFWX7q53wvLEJv0PoyPvBX5v07bc6Vsb8KyVv5NXpQQQCy0a90er6VKkUJOrgCW0m2yoy6NSMImiaHg3kT+bWUT3w7gJJWD2WfMuYWJ+oilculek3KI1l6VK0oCJ1iF4WTaqx3gpQIja1FKusNd1Z/55SvUDPNE7Xbn7ktY9cUzfw7JsJK+X9mPVaRP/gxeM7St8CwsNsX8KjazrY5XTTxUZ0+sU3fjB+WmMuUlT3m5vuQ++mkfGL6lBhYSvrKWKDBdvdV/ccM9Inqz89VX9fe/UdGtzkz4Cijaz0Eoad33wZnGQSERB2xTAJUV3QMXtYM4jfl/eDp+jyqngcynMKST+rwtD2qzvGPJJQapr17Sst/Sl+F5mPqGX1aD16IQU35ml0G1D+5Cj+A/+6qT8IVdIG8QnuNNZoEFCNGC+plb3wSC39zLUQ5g2sG1pH6LPqTlCh3QKsyhSEWOc1qR81VRF5Kz47K2NcyVEWVabFE15Jf1DcIbLZl1n1z45qIdwEtILShakcwqDrVwloW/IifvUy+ANRd4UoXic0fRQFcxeAf7JRnsX7Zay5vTsBpu2259EnDIjytbaI7RbifQNnq9tWFO92UU4VpnW52UY6yaYYOSl0su/vUF7He8k2L0qarAyTcN5gJFFYBpyTCPR92z3WA7rKFwlck/fbgQ+RboU0gYJCIuvongJgs1eiTPYO56j0gGTTmJx+TgdzDnEDgxJ3uUXUkUf501FV2yCmpWD168wXUp0gGCpmBmE/cOhtveuiGvGwuiJhbrxeHrOHqWOAikui81WNtrPJjN217KMVQXCDL+G8iRcLTcfV9E75Uc2Itu9ISXqPlshvZLrkAHWMwf3lAD71wn4T1ouukl4h2PWMQYmEzxnst0VV1Nrx6PQsLOS1TXMerrwE+owTGp8VA/8i6IODZVFo/Ib+HLcTqlq+C+JDhi6xfMgBHWEri65POoecjU/ICJgwFeQE70iwi2eWZilJfB32lcMZ6xRbBWYAd6pLC0JsYw0Vpc3wgPz12TDsKZquEe9iUAbESdZenQV2FtaP/GuhOp5XbrL9v3W+F7PtB+QRspTgfBnGHR4+SAIrUF8NRrS3f1grZyORAw4BakfZnVKfaYD7/+JFIPew5UAki58Og2fBFuza+txhwOcIeEd+kvJMbHrin67iDkrkOS/bNFSjGijv5L2Xoz6dxZCmnLZaVgiLD2ZimWb1nsTCyfD1rahdCQ1JI+DYgbqzrToYaBPTza3JP1lnpIlEXXaNrBnqK4/jZaHGnoXo0gTXtPw34Hr8uIqSRiXejR4uj7GqBJ90nYtvvyBmjaeRC27NwwW9ZpTLfEDjfBZRsOg4S4i+19n673Fxm0kOFZqLIWOwQDMlBf3kZjoBNFyFiXvAd7fqPRjlEW/Au/aQoTBxTmzT5Jywtq6Exk+Da6r/YtFfhvmHKyp9gyDU75P/gf70zcPQ==')).decode()
FEATURE_IDENTITY = dict(runtime=RUNTIME_VERSION, encoder_sha256=ENCODER_HASH, backend=BACKEND,
    dataset_identity=INDEX['identity'], representation=CFG['representation'], input_size=322,
    feature_pool='cls_support_weighted_patch_mean', candidates='all_acquired',
    torch_version=torch.__version__, numpy_version=np.__version__,
    encoder_library=importlib.metadata.version('transformers' if BACKEND=='hf' else 'timm'),
    code_sha256=hashlib.sha256((''.join(definition_source(name) for name in
        ['read_native','FrozenEncoder','make_slice_inputs','protocol_codes','extract_features'])).encode()).hexdigest())
TRAINING_ID = fingerprint(dict(cfg=CFG, feature=FEATURE_IDENTITY,
    runtime_sha256=hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest()))
WORK = OUTPUT_ROOT/TRAINING_ID[:16]
WORK.mkdir(parents=True, exist_ok=True)
atomic_json(WORK/'run_identity.json', dict(training_id=TRAINING_ID, feature_id=fingerprint(FEATURE_IDENTITY)))
atomic_json(WORK/'configuration.json', CFG)
atomic_json(WORK/'preprocessing_identity.json', INDEX)
audit_labels(LABELS).to_csv(WORK/'label_audit_private.csv', index=False)
if not RESUME_ROOTS:
    RESUME_ROOTS = [p.parent for p in find_input_files('/kaggle/input','run_identity.json')
                    if json.loads(p.read_text()).get('feature_id') == fingerprint(FEATURE_IDENTITY)]
RESUME_ROOTS = [Path(p) for p in RESUME_ROOTS]
for p in RESUME_ROOTS:
    if not (p/'run_identity.json').is_file(): raise FileNotFoundError('Resume folder needs run_identity.json: '+str(p))
HEAD_RESUME_ROOTS = [p for p in RESUME_ROOTS if json.loads((p/'run_identity.json').read_text()).get('training_id') == TRAINING_ID]
print('Private run folder:', WORK)
print('Compatible feature sources:', len(RESUME_ROOTS), '| head sources:', len(HEAD_RESUME_ROOTS))


## 5. Frozen feature extraction
Features are cached per series, atomically. All valid slice centers are encoded once; each training batch samples up to 16. This can require multiple sessions. Save partial output privately and reattach it; do not bypass missing-shard checks. No claim is made that the entire dataset fits one GPU session.

In [ ]:
if MODE == 'train':
    fid = fingerprint(FEATURE_IDENTITY)
    sources = [WORK/'features'/fid] + [p/'features'/fid for p in RESUME_ROOTS]
    paths = []
    for r in SERIES.to_dict('records'):
        path = next((p/feature_name(r) for p in sources if (p/feature_name(r)).is_file()), None)
        if path is None: raise FileNotFoundError('Feature cache incomplete; run MODE=features or all')
        check_features(path,r,fid)
        target = WORK/'features'/fid/feature_name(r)
        target.parent.mkdir(parents=True,exist_ok=True)
        if path.resolve() != target.resolve():
            if shutil.disk_usage(WORK).free < CFG['min_free_gb']*1e9 + path.stat().st_size:
                raise OSError('Insufficient space to retain cumulative feature output')
            tmp = target.with_suffix('.tmp'); shutil.copy2(path,tmp); tmp.replace(target)
        paths.append(str(target))
    FEATURE_SERIES = SERIES.copy(); FEATURE_SERIES['feature_path'] = paths
else:
    FEATURE_SERIES = extract_features(SERIES,ENCODER,INDEX['identity'],FEATURE_IDENTITY,WORK,
                                      RESUME_ROOTS,CFG,BUDGET)
READY = FEATURE_SERIES is not None
# Keep encoder on CPU while the small heads train; release GPU allocator reservations.
ENCODER = ENCODER.cpu()
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
if READY:
    atomic_json(WORK/'status.json',dict(status='FEATURES_COMPLETE',training_id=TRAINING_ID))
    print('Feature cache complete. Head training can now run without image decoding.')


## 6. Train and validate fold heads
Unknown labels contribute no loss. Metadata scalers use training folds only. The primary objective averages weighted BCE equally across supervised targets. Current and EMA weights compete on verified validation loss; best checkpoints are retained. Ranking loss and verified-only refinement are opt-in. Training checkpoints preserve the last complete epoch; an interruption restarts an unfinished epoch.

In [ ]:
RESULTS = []
if READY and MODE != 'features':
    DATA = FeatureStudies(FEATURE_SERIES,LABELS,CFG['max_feature_ram_gb'])
    for fold in FOLDS:
        result = train_fold(DATA,fold,CFG,WORK,HEAD_RESUME_ROOTS,TRAINING_ID,BUDGET,DEVICE)
        if result[0] is None:
            atomic_json(WORK/'status.json',dict(status='TRAINING_PARTIAL',training_id=TRAINING_ID,
                        completed_folds=[r[0]['fold'] for r in RESULTS]))
            print('Training paused at session budget. Reattach this private output to resume.')
            break
        RESULTS.append(result)
else:
    print('Head training skipped: feature-only mode or incomplete features.')


## 7. Export and verify
Only a completed requested fold set is exported. A one-fold pilot is marked as incomplete OOF coverage. The package includes a shared encoder, heads, scalers, manifests, exact runtime, native preprocessing functions/configuration, environment versions, and a synthetic head reload test. Private features and OOF rows stay outside the package. No submission file is generated by training.

In [ ]:
if READY and MODE != 'features' and len(RESULTS) == len(FOLDS):
    PACKAGE, METRICS = export_package(ENCODER,DATA,RESULTS,CFG,INDEX['identity'],FEATURE_IDENTITY,
        TRAINING_ID,WORK,RUNTIME_SOURCE,PREPROCESSING_SOURCE,INDEX['preprocessing'])
    # Test strict head reload against deterministic synthetic inputs.
    reference = torch.load(PACKAGE/'synthetic_head_smoke.pt',map_location='cpu',weights_only=True)
    saved = torch.load(PACKAGE/f"head_fold_{RESULTS[0][0]['fold']}.pt",map_location='cpu',weights_only=True)
    reloaded = FindingAttention(saved['hidden']).eval(); reloaded.load_state_dict(saved['state_dict'],strict=True)
    with torch.inference_mode():
        actual = reloaded(reference['x'],reference['protocol'],reference['spacing'],reference['mask'])
    torch.testing.assert_close(actual,reference['expected'],rtol=1e-5,atol=1e-6)
    manifest = json.loads((PACKAGE/'manifest.json').read_text())
    for name,digest in manifest['files'].items():
        if file_hash(PACKAGE/name) != digest: raise ValueError('Export checksum mismatch: '+name)
    atomic_json(WORK/'status.json',dict(status='COMPLETE_REQUESTED_FOLDS',training_id=TRAINING_ID,
                complete_oof=METRICS['complete_oof'],folds=FOLDS))
    display(pd.DataFrame(METRICS['pooled']['targets']))
    print('OOF macro AUROC:',METRICS['pooled']['macro_auroc'])
    print('Complete eligible OOF coverage:',METRICS['complete_oof'])
    print('Model package:',PACKAGE)
    print('Keep the full notebook output private. This run has not submitted or published anything.')


## Appendix: terms and limitations

| Term | Meaning |
|---|---|
| Study / series / slice | Examination / acquisition / one 2D image. |
| Frozen encoder | Image model whose parameters remain unchanged. |
| CLS / patch features | Whole-input summary / representations of small image regions. |
| Pooling | Combining slice features into a series representation. |
| Finding query | Learned attention request for one abnormality. |
| Label mask / support mask | Known diagnosis indicator / acquired-pixel indicator. |
| BCE / logit | Binary prediction loss / raw prediction before sigmoid. |
| EMA | Exponential moving average of model weights. |
| OOF | Predictions on each model's held-out fold. |
| AUROC / average precision | Ranking discrimination / a precision–recall summary. |
| Provenance | Record of model, data and preprocessing origins. |

The model uses study-level supervision, not lesion segmentation. Attention weights are not verified localization. Patch support weighting does not prevent padding from affecting the encoder internally. Undefined single-class AUROCs are recorded as unavailable. Report per-target coverage and results; our accepted generated labels do not provide equal support for all targets. There is no encoder fine-tuning or bootstrap uncertainty analysis in this initial notebook; both are later experiments.

References: local `training-algorithm.md`; Roman Tamrazov and evgendvorkin notebooks in `resources/reference_notebooks/`; [DINOv2 model card](https://github.com/facebookresearch/dinov2/blob/main/MODEL_CARD.md); [Transformers DINOv2 interface](https://huggingface.co/docs/transformers/model_doc/dinov2); [attention-based MIL](https://proceedings.mlr.press/v80/ilse18a.html).
